# Data Preparation

## Set Up Environment

### Install Dependencies

In [67]:
!pip install --upgrade pandas sentence-transformers pandas tqdm urllib3

### Import Dependencies

In [69]:
import pandas as pd
import numpy as np
import re
import os

from sentence_transformers import SentenceTransformer
from tqdm import tqdm
from urllib.parse import urlparse

### Load Model Sentence-BERT (IndoSBERT)

In [3]:
model_name = 'denaya/indoSBERT-large'

model = SentenceTransformer(model_name)

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 12904.54it/s]


### Load Raw Data

In [16]:
data_path = 'datasets/raw_tickets.csv'

df_raw = pd.read_csv(data_path)

df = df_raw[['DESKRIPSI']].dropna()
df.rename(columns={'DESKRIPSI': 'RAW TICKET'}, inplace=True)

df.head()

,RAW TICKET
0,Klien: Setwapres Medsos\nisu: Crawlback Commen...
1,"selamat pagi tim it, mohon bantuannya saya men..."
2,Klien: BPS\nIsu: Data postingan Instagram tida...
3,selamat sore mas @Dhanysybn dan tim info untuk...
4,Klien: Heinz \nIsu: Dashboard loading\n\nSelam...


## Data Preprocessing

### Preprocessed Data for Sentence-Embedding Model

#### Delete Duplicate Data

In [20]:
# hapus data yang duplikat
print(f"Jumlah data sebelum dihapus duplikat: {len(df)}")

df.drop_duplicates(inplace=True)

print(f"Jumlah data setelah dihapus duplikat: {len(df)}")

Jumlah data sebelum dihapus duplikat: 1639
Jumlah data setelah dihapus duplikat: 1622


#### Replace new line special character

In [22]:
# jika terdapat lebih dari satu newline, ganti dahulu menjadi satu newline, lalu ganti newline tersebut dengan titik dan spasi
df['REPLACED NEWLINE'] = df['RAW TICKET'].apply(lambda x: re.sub(r'\n+', '. ', x))

df.head()

,RAW TICKET,REPLACED NEWLINE
0,Klien: Setwapres Medsos\nisu: Crawlback Commen...,Klien: Setwapres Medsos. isu: Crawlback Commen...
1,"selamat pagi tim it, mohon bantuannya saya men...","selamat pagi tim it, mohon bantuannya saya men..."
2,Klien: BPS\nIsu: Data postingan Instagram tida...,Klien: BPS. Isu: Data postingan Instagram tida...
3,selamat sore mas @Dhanysybn dan tim info untuk...,selamat sore mas @Dhanysybn dan tim info untuk...
4,Klien: Heinz \nIsu: Dashboard loading\n\nSelam...,Klien: Heinz . Isu: Dashboard loading. Selamat...


#### Lower Casing

In [23]:
df['LOWERCASE'] = df['REPLACED NEWLINE'].str.lower()

df.head()

,RAW TICKET,REPLACED NEWLINE,LOWERCASE
0,Klien: Setwapres Medsos\nisu: Crawlback Commen...,Klien: Setwapres Medsos. isu: Crawlback Commen...,klien: setwapres medsos. isu: crawlback commen...
1,"selamat pagi tim it, mohon bantuannya saya men...","selamat pagi tim it, mohon bantuannya saya men...","selamat pagi tim it, mohon bantuannya saya men..."
2,Klien: BPS\nIsu: Data postingan Instagram tida...,Klien: BPS. Isu: Data postingan Instagram tida...,klien: bps. isu: data postingan instagram tida...
3,selamat sore mas @Dhanysybn dan tim info untuk...,selamat sore mas @Dhanysybn dan tim info untuk...,selamat sore mas @dhanysybn dan tim info untuk...
4,Klien: Heinz \nIsu: Dashboard loading\n\nSelam...,Klien: Heinz . Isu: Dashboard loading. Selamat...,klien: heinz . isu: dashboard loading. selamat...


#### Remove Klien and Placeholder 'Isu/Kendala'

In [ ]:
def bersihkan_tiket_revisi(teks):
    if not isinstance(teks, str):
        return teks
    
    # 1. MENGHAPUS BLOK KLIEN
    # Grup 1 (klien, client): Pasti placeholder, hapus beserta titik/koma setelahnya
    # Grup 2 (dashboard, project): HANYA dianggap placeholder jika diikuti titik dua/titik koma (: atau ;)
    pattern_klien = r'\b(?:(?:klien|client|clien|lien)[\s:;,\.]*|(?:dashboard trial|dashboard client|dashboard|project)\s*[:;]\s*)(.*?)(?=\b(?:isu|issue|kendala|request|req|keluhan|problem|kebutuhan|detail)\b|$)'
    teks = re.sub(pattern_klien, '', teks)
    
    # 2. MENGHAPUS LABEL ISU & DETAIL
    # SAMA SEPERTI GRUP 2: HANYA dihapus jika bertindak sebagai label (diikuti titik dua/titik koma)
    # Ini menyelamatkan kata "isu" atau "keluhan" yang berada di tengah/akhir kalimat.
    pattern_isu_detail = r'\b(?:isu|issue|kendala|request|req|keluhan|problem|kebutuhan|detail)\s*[:;]\s*'
    teks = re.sub(pattern_isu_detail, '', teks)
    
    # 3. FINALISASI
    teks = teks.strip(' ;:,-') # Titik tidak dimasukkan agar titik akhir kalimat aman
    
    # Menghapus spasi ganda
    teks = re.sub(r'\s+', ' ', teks)
    
    return teks

# Terapkan fungsinya
df['CLEANED TICKET'] = df['LOWERCASE'].apply(bersihkan_tiket_revisi)
df.head()

,RAW TICKET,REPLACED NEWLINE,LOWERCASE,CLEANED TICKET
0,Klien: Setwapres Medsos\nisu: Crawlback Commen...,Klien: Setwapres Medsos. isu: Crawlback Commen...,klien: setwapres medsos. isu: crawlback commen...,crawlback comment. siang tim it minta tolong b...
1,"selamat pagi tim it, mohon bantuannya saya men...","selamat pagi tim it, mohon bantuannya saya men...","selamat pagi tim it, mohon bantuannya saya men...","selamat pagi tim it, mohon bantuannya saya men..."
2,Klien: BPS\nIsu: Data postingan Instagram tida...,Klien: BPS. Isu: Data postingan Instagram tida...,klien: bps. isu: data postingan instagram tida...,data postingan instagram tidak masuk. selamat ...
3,selamat sore mas @Dhanysybn dan tim info untuk...,selamat sore mas @Dhanysybn dan tim info untuk...,selamat sore mas @dhanysybn dan tim info untuk...,selamat sore mas @dhanysybn dan tim info untuk...
4,Klien: Heinz \nIsu: Dashboard loading\n\nSelam...,Klien: Heinz . Isu: Dashboard loading. Selamat...,klien: heinz . isu: dashboard loading. selamat...,"dashboard loading. selamat sore tim it, mohon ..."


#### Masking Nama dan Alamat Email

In [27]:
# Masking nama akun (contoh: @namaakun) dengan placeholder nama orang
df['MASKED TICKET'] = df['CLEANED TICKET'].apply(lambda x: re.sub(r'@\w+', 'nama orang', x))

# Masking email (contoh: email@domain.com) dengan placeholder email
df['MASKED TICKET'] = df['MASKED TICKET'].apply(lambda x: re.sub(r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Z|a-z]{2,}\b', 'email@domain.com', x))

df.head()

,RAW TICKET,REPLACED NEWLINE,LOWERCASE,CLEANED TICKET,MASKED TICKET
0,Klien: Setwapres Medsos\nisu: Crawlback Commen...,Klien: Setwapres Medsos. isu: Crawlback Commen...,klien: setwapres medsos. isu: crawlback commen...,crawlback comment. siang tim it minta tolong b...,crawlback comment. siang tim it minta tolong b...
1,"selamat pagi tim it, mohon bantuannya saya men...","selamat pagi tim it, mohon bantuannya saya men...","selamat pagi tim it, mohon bantuannya saya men...","selamat pagi tim it, mohon bantuannya saya men...","selamat pagi tim it, mohon bantuannya saya men..."
2,Klien: BPS\nIsu: Data postingan Instagram tida...,Klien: BPS. Isu: Data postingan Instagram tida...,klien: bps. isu: data postingan instagram tida...,data postingan instagram tidak masuk. selamat ...,data postingan instagram tidak masuk. selamat ...
3,selamat sore mas @Dhanysybn dan tim info untuk...,selamat sore mas @Dhanysybn dan tim info untuk...,selamat sore mas @dhanysybn dan tim info untuk...,selamat sore mas @dhanysybn dan tim info untuk...,selamat sore mas nama orang dan tim info untuk...
4,Klien: Heinz \nIsu: Dashboard loading\n\nSelam...,Klien: Heinz . Isu: Dashboard loading. Selamat...,klien: heinz . isu: dashboard loading. selamat...,"dashboard loading. selamat sore tim it, mohon ...","dashboard loading. selamat sore tim it, mohon ..."


#### Remove Emoticon

In [29]:
# Hapus emotikon
def hapus_emotikon(teks):
    if not isinstance(teks, str):
        return teks
    # Rentang emotikon umum (emoji, simbol, dll.)
    emotikon_pattern = r'[\U0001F600-\U0001F64F\U0001F300-\U0001F5FF\U0001F680-\U0001F6FF\U0001F700-\U0001F77F\U0001F780-\U0001F7FF\U0001F800-\U0001F8FF\U0001F900-\U0001F9FF\U0001FA00-\U0001FA6F\U0001FA70-\U0001FAFF]+'
    return re.sub(emotikon_pattern, '', teks)

df['REMOVED EMOJIS'] = df['MASKED TICKET'].apply(hapus_emotikon)
df.head()

,RAW TICKET,REPLACED NEWLINE,LOWERCASE,CLEANED TICKET,MASKED TICKET,REMOVED EMOJIS
0,Klien: Setwapres Medsos\nisu: Crawlback Commen...,Klien: Setwapres Medsos. isu: Crawlback Commen...,klien: setwapres medsos. isu: crawlback commen...,crawlback comment. siang tim it minta tolong b...,crawlback comment. siang tim it minta tolong b...,crawlback comment. siang tim it minta tolong b...
1,"selamat pagi tim it, mohon bantuannya saya men...","selamat pagi tim it, mohon bantuannya saya men...","selamat pagi tim it, mohon bantuannya saya men...","selamat pagi tim it, mohon bantuannya saya men...","selamat pagi tim it, mohon bantuannya saya men...","selamat pagi tim it, mohon bantuannya saya men..."
2,Klien: BPS\nIsu: Data postingan Instagram tida...,Klien: BPS. Isu: Data postingan Instagram tida...,klien: bps. isu: data postingan instagram tida...,data postingan instagram tidak masuk. selamat ...,data postingan instagram tidak masuk. selamat ...,data postingan instagram tidak masuk. selamat ...
3,selamat sore mas @Dhanysybn dan tim info untuk...,selamat sore mas @Dhanysybn dan tim info untuk...,selamat sore mas @dhanysybn dan tim info untuk...,selamat sore mas nama orang dan tim info untuk...,selamat sore mas nama orang dan tim info untuk...,selamat sore mas nama orang dan tim info untuk...
4,Klien: Heinz \nIsu: Dashboard loading\n\nSelam...,Klien: Heinz . Isu: Dashboard loading. Selamat...,klien: heinz . isu: dashboard loading. selamat...,"dashboard loading. selamat sore tim it, mohon ...","dashboard loading. selamat sore tim it, mohon ...","dashboard loading. selamat sore tim it, mohon ..."


#### Parse Url

In [71]:
def ubah_url_ke_domain_revisi(teks):
    if not isinstance(teks, str):
        return teks
    
    # POLA REGEX BARU: 
    # Menangkap seluruh string URL yang valid (termasuk huruf, angka, strip, titik, dan garis miring)
    pola_url = r'https?://[\w\-\.\/\?\&\=\%]+'
    
    def ekstrak_domain(match):
        url = match.group(0)
        
        # Mencegah titik atau koma di akhir kalimat ikut terbaca sebagai bagian URL
        if url.endswith('.') or url.endswith(','):
            url = url[:-1]
            
        try:
            # urlparse akan membedah URL. 
            # Contoh: dari "https://megapolitan.kompas.com/..." kita ambil "megapolitan.kompas.com"
            netloc = urlparse(url).netloc
            
            # Pecah berdasarkan titik
            parts = netloc.split('.')
            
            # LOGIKA PENCARIAN NAMA DOMAIN UTAMA:
            # Kasus 1: Domain Indonesia 3 tingkat (contoh: news.detik.co.id -> ambil 'detik')
            if len(parts) >= 3 and parts[-2] in ['co', 'go', 'ac', 'or', 'sch', 'my']:
                domain_utama = parts[-3]
                
            # Kasus 2: Domain standar dengan/tanpa subdomain (contoh: megapolitan.kompas.com -> ambil 'kompas')
            elif len(parts) >= 2:
                domain_utama = parts[-2]
                
            # Kasus 3: Fallback jika format URL tidak biasa
            else:
                domain_utama = parts[0]
                
            # Jika domain utama tertangkap sebagai 'www', ambil kata setelahnya
            if domain_utama == 'www' and len(parts) >= 2:
                domain_utama = parts[-1]
                
            return f"tautan {domain_utama.lower()}"
            
        except Exception:
            return "tautan" # Fallback jika terjadi error parsing
            
    # Terapkan re.sub menggunakan fungsi ekstrak_domain
    teks = re.sub(pola_url, ekstrak_domain, teks)
    
    return teks

# Terapkan fungsi ke kolom yang sudah diproses sebelumnya
df['FINAL TICKET'] = df['REMOVED EMOJIS'].apply(ubah_url_ke_domain_revisi)
df.head()

,RAW TICKET,REPLACED NEWLINE,LOWERCASE,CLEANED TICKET,MASKED TICKET,REMOVED EMOJIS,FINAL TICKET
0,Klien: Setwapres Medsos\nisu: Crawlback Commen...,Klien: Setwapres Medsos. isu: Crawlback Commen...,klien: setwapres medsos. isu: crawlback commen...,crawlback comment. siang tim it minta tolong b...,crawlback comment. siang tim it minta tolong b...,crawlback comment. siang tim it minta tolong b...,crawlback comment. siang tim it minta tolong b...
1,"selamat pagi tim it, mohon bantuannya saya men...","selamat pagi tim it, mohon bantuannya saya men...","selamat pagi tim it, mohon bantuannya saya men...","selamat pagi tim it, mohon bantuannya saya men...","selamat pagi tim it, mohon bantuannya saya men...","selamat pagi tim it, mohon bantuannya saya men...","selamat pagi tim it, mohon bantuannya saya men..."
2,Klien: BPS\nIsu: Data postingan Instagram tida...,Klien: BPS. Isu: Data postingan Instagram tida...,klien: bps. isu: data postingan instagram tida...,data postingan instagram tidak masuk. selamat ...,data postingan instagram tidak masuk. selamat ...,data postingan instagram tidak masuk. selamat ...,data postingan instagram tidak masuk. selamat ...
3,selamat sore mas @Dhanysybn dan tim info untuk...,selamat sore mas @Dhanysybn dan tim info untuk...,selamat sore mas @dhanysybn dan tim info untuk...,selamat sore mas nama orang dan tim info untuk...,selamat sore mas nama orang dan tim info untuk...,selamat sore mas nama orang dan tim info untuk...,selamat sore mas nama orang dan tim info untuk...
4,Klien: Heinz \nIsu: Dashboard loading\n\nSelam...,Klien: Heinz . Isu: Dashboard loading. Selamat...,klien: heinz . isu: dashboard loading. selamat...,"dashboard loading. selamat sore tim it, mohon ...","dashboard loading. selamat sore tim it, mohon ...","dashboard loading. selamat sore tim it, mohon ...","dashboard loading. selamat sore tim it, mohon ..."


#### Save Results to CSV

In [72]:
# Simpan hasil akhir ke CSV baru
output_path = 'datasets/cleaned_tickets.csv'

df_result = df[['RAW TICKET', 'FINAL TICKET']].copy()
df_result.rename(columns={'FINAL TICKET': 'CLEANED TICKET'}, inplace=True)
df_result.to_csv(output_path, index=False)

### Sentence-Embedding

#### Implements Sliding Windows

In [73]:
tokenizer = model.tokenizer

def get_sliding_window_embedding_silent(teks, model, max_length=256, stride=50):
    # 1. Simpan batas asli dan ubah batas maksimal tokenizer sementara
    # Ini membungkam warning "450 > 256" dari HuggingFace
    original_max_length = tokenizer.model_max_length
    tokenizer.model_max_length = 100000 
    
    # Lakukan tokenisasi utuh tanpa warning
    tokens = tokenizer.encode(teks, add_special_tokens=False)
    
    # Kembalikan batas tokenizer ke aslinya
    tokenizer.model_max_length = original_max_length
    
    # 2. Jika teks pendek, langsung encode biasa (dikurangi 2 untuk [CLS] dan [SEP])
    if len(tokens) <= max_length - 2:
        return model.encode(teks)
    
    # 3. Persiapan Sliding Window
    chunk_embeddings = []
    window_size = max_length - 2 
    
    for i in range(0, len(tokens), window_size - stride):
        chunk_tokens = tokens[i : i + window_size]
        chunk_text = tokenizer.decode(chunk_tokens)
        
        # Passing chunk teks ke model
        chunk_emb = model.encode(chunk_text)
        chunk_embeddings.append(chunk_emb)
        
        if i + window_size >= len(tokens):
            break
            
    # 4. Agregasi menggunakan Mean Pooling
    final_embedding = np.mean(chunk_embeddings, axis=0)
    
    return final_embedding

#### Embed Sentence

In [74]:
# Mengaktifkan ekstensi pandas dari tqdm untuk memunculkan progress bar
tqdm.pandas(desc="Proses Embedding IndoSBERT")

# 1. Mencegah Error: Pastikan tidak ada data kosong (NaN) akibat proses pembersihan sebelumnya
# Mengubah NaN atau nilai kosong menjadi string kosong
df_result['CLEANED TICKET'] = df_result['CLEANED TICKET'].fillna("").astype(str)

# 2. Ekstraksi Embedding dengan Sliding Window
# Kita gunakan .progress_apply() sebagai pengganti .apply() biasa agar progress bar muncul
df_result['EMBEDDING'] = df_result['CLEANED TICKET'].progress_apply(
    lambda teks: get_sliding_window_embedding_silent(
        teks=teks, 
        model=model, 
        max_length=256, 
        stride=128
    )
)

# Menampilkan 5 baris pertama untuk memastikan kolom EMBEDDING sudah terbentuk
df_result.head()

Proses Embedding IndoSBERT:   1%|          | 12/1622 [00:05<12:35,  2.13it/s]


KeyboardInterrupt: 

In [75]:
tqdm.pandas(desc="Proses Embedding IndoSBERT")

# embedding tanpa sliding window
df_result['EMBEDDING'] = df_result['CLEANED TICKET'].progress_apply(lambda teks: model.encode(teks))

df_result.head()

Proses Embedding IndoSBERT: 100%|██████████| 1622/1622 [11:40<00:00,  2.32it/s]


,RAW TICKET,CLEANED TICKET,EMBEDDING
0,Klien: Setwapres Medsos\nisu: Crawlback Commen...,crawlback comment. siang tim it minta tolong b...,"[0.13826014, -0.5070506, 0.01965604, 0.1804346..."
1,"selamat pagi tim it, mohon bantuannya saya men...","selamat pagi tim it, mohon bantuannya saya men...","[0.18187888, -0.007482365, -0.53676426, -0.079..."
2,Klien: BPS\nIsu: Data postingan Instagram tida...,data postingan instagram tidak masuk. selamat ...,"[-0.24981816, -0.0044697057, -0.46794048, 0.31..."
3,selamat sore mas @Dhanysybn dan tim info untuk...,selamat sore mas nama orang dan tim info untuk...,"[0.17636633, 0.06786756, -0.08733067, 0.348039..."
4,Klien: Heinz \nIsu: Dashboard loading\n\nSelam...,"dashboard loading. selamat sore tim it, mohon ...","[0.25090253, -0.092597604, -0.030185038, 0.220..."


#### Save to Pickle

In [76]:
output_path = 'datasets/tickets.pkl'

df_result.to_pickle(output_path)

In [77]:
df_result[['CLEANED TICKET']].to_csv('datasets/preprocessed_tickets.csv', index=False)